# ⚽ Análisis Exploratorio de Datos (EDA) - Premier League 2025-26
Este notebook contiene el análisis inicial de los datasets de partidos y eventos para la construcción de los modelos de xG y Match Predictor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mplsoccer import Pitch
import json

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('viridis')

## 1. Carga de Datos

In [ ]:
df_matches = pd.read_csv('../data/matches.csv')
df_events = pd.read_csv('../data/events.csv')
df_players = pd.read_csv('../data/players.csv')

print(f"Matches: {df_matches.shape}")
print(f"Events: {df_events.shape}")
print(f"Players: {df_players.shape}")

## 2. EDA: Goles Esperados (xG)
Análisis de la ubicación de los tiros y su efectividad.

In [ ]:
# Filtrar solo por tiros (is_shot == True)
df_shots = df_events[df_events['is_shot'] == True].copy()

print(f"Total de tiros en el dataset: {len(df_shots)}")
print(f"Total de goles: {df_shots['is_goal'].sum()}")
print(f"Conversión promedio: {df_shots['is_goal'].mean():.2%}")

In [ ]:
# Visualizar la ubicación de los tiros en la cancha
pitch = Pitch(pitch_type='opta', pitch_color='#22312b', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(10, 7))

# Tiros que fueron gol (Azul) vs Fallados (Rojo)
pitch.scatter(df_shots[df_shots['is_goal'] == False]['x'], 
              df_shots[df_shots['is_goal'] == False]['y'], 
              ax=ax, color='red', alpha=0.3, label='Fallado')
pitch.scatter(df_shots[df_shots['is_goal'] == True]['x'], 
              df_shots[df_shots['is_goal'] == True]['y'], 
              ax=ax, color='cyan', edgecolors='white', s=100, label='Gol')

ax.legend(facecolor='#22312b', edgecolor='None', fontsize=12, labelcolor='white')
plt.title('Mapa de Tiros - Premier League 2025-26', color='black', fontsize=15)
plt.show()

## 3. EDA: Predicción de Partidos

In [ ]:
# Distribución de resultados (Home, Draw, Away)
plt.figure(figsize=(8, 5))
sns.countplot(data=df_matches, x='ftr', order=['H', 'D', 'A'])
plt.title('Distribución de Resultados Finales (Ftr)')
plt.xlabel('Resultado (Local, Empate, Visitante)')
plt.ylabel('Cantidad de Partidos')
plt.show()

print(df_matches['ftr'].value_counts(normalize=True))

## 4. Feature Engineering: Contexto y Calificadores
Extracción de calificadores y cálculo de métricas geométricas/situacionales.

In [ ]:
def parse_shot_qualifiers(qual_str):
    try:
        quals = json.loads(qual_str)
        q_ids = [q['type']['value'] for q in quals]
        return {
            'is_big_chance': 214 in q_ids,
            'is_header': 15 in q_ids,
            'is_counter': 23 in q_ids,
            'is_penalty': 9 in q_ids,
            'is_free_kick': 26 in q_ids
        }
    except:
        return {'is_big_chance': False, 'is_header': False, 'is_counter': False, 'is_penalty': False, 'is_free_kick': False}

quals_df = df_shots['qualifiers'].apply(parse_shot_qualifiers).apply(pd.Series)
df_shots = pd.concat([df_shots, quals_df], axis=1)

df_shots['distance'] = np.sqrt((100 - df_shots['x'])**2 + (50 - df_shots['y'])**2)
df_shots['angle'] = np.arctan2(abs(50 - df_shots['y']), (100 - df_shots['x']))

### 4.1 Análisis Situacional: Game State y Minutos Críticos

In [ ]:
# Calcular el marcador en vivo para cada evento
df_events['is_goal_int'] = df_events['is_goal'].astype(int)
df_events['team_score'] = df_events.groupby(['match_id', 'team_name'])['is_goal_int'].cumsum() - df_events['is_goal_int']
df_events['match_total_goals'] = df_events.groupby('match_id')['is_goal_int'].cumsum() - df_events['is_goal_int']
df_events['opp_score'] = df_events['match_total_goals'] - df_events['team_score']
df_events['score_diff'] = df_events['team_score'] - df_events['opp_score']

df_shots = df_shots.merge(df_events[['id', 'score_diff']], on='id', how='left')
df_shots['is_late_half'] = df_shots['minute'].apply(lambda x: 1 if (x >= 40 and x <= 45) or (x >= 85) else 0)

### 4.2 Incorporando la Posición del Jugador

In [ ]:
# Unir tiros con tabla de jugadores para obtener la posición
df_shots = df_shots.merge(df_players[['id', 'position']], left_on='player_id', right_on='id', how='left')

# Visualizar eficiencia por posición
pos_efficiency = df_shots.groupby('position')['is_goal'].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
pos_efficiency.plot(kind='bar', color='coral')
plt.title('Efectividad de Tiro según Posición del Jugador')
plt.ylabel('Tasa de Conversión (is_goal)')
plt.show()

print("Tasa de conversión por posición:")
print(pos_efficiency)